# Mục 3b(ii) — Làm mịn platelets bằng mean của 5 nhóm
**Võ Minh Huy — 2311110117**

Thực hiện cùng phép làm mịn với SQL 05b. Đầu vào là bản làm việc sau N07 do notebook 11 đã lưu; không lấy dữ liệu đã làm mịn để tính lại. Chỉ cập nhật platelets trên bản sao riêng. Phần tăng/giảm 10% là bước tiếp theo.

## 1. Chuẩn bị đầu vào
Chia miền giá trị thành 5 khoảng có độ rộng bằng nhau. Dùng Decimal để tính mean và làm tròn nhất quán với SQL.

In [1]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP, ROUND_FLOOR, localcontext
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_sau_n07 = thu_muc / "ket_qua" / "11_dien_ejection_fraction" / "ban_lam_viec_sau_ef_python.csv"
if not file_sau_n07.exists():
    raise FileNotFoundError("Chạy Run All notebook 11_dien_null_ejection_fraction.ipynb trước.")
ban_dau = pd.read_csv(file_sau_n07, dtype=str, keep_default_na=False)
ban_dau["MaHoSo"] = ban_dau["MaHoSo"].astype(int)
ban_dau["platelets"] = ban_dau["platelets"].map(Decimal)

so_nhom = 5
min_p = ban_dau["platelets"].min()
max_p = ban_dau["platelets"].max()
if max_p <= min_p:
    raise ValueError("Miền platelets không đủ để chia nhóm.")
do_rong = (max_p - min_p) / so_nhom
pd.DataFrame({"SoHoSo":[len(ban_dau)], "Min":[min_p], "Max":[max_p],
              "SoNhom":[so_nhom], "DoRongNhom":[do_rong]})

,SoHoSo,Min,Max,SoNhom,DoRongNhom
0,299,25100.000000,850000.000000,5,164980.000000


## 2. Chia 5 nhóm và làm mịn bằng mean
Khoảng trái đóng, phải mở; riêng nhóm 5 lấy cả Max. Nhóm = floor((x − Min) / độ rộng) + 1; riêng x = Max gán nhóm 5. Tính mean một lần từ giá trị trước làm mịn, lưu 10 số lẻ rồi làm tròn giá trị thay thế đến 2 chữ số theo ROUND_HALF_UP.

In [2]:
def xac_dinh_nhom(value):
    if value == max_p:
        return so_nhom
    return 1 + int(((value - min_p) / do_rong).to_integral_value(rounding=ROUND_FLOOR))

nhat_ky = pd.DataFrame({"MaHoSo":ban_dau["MaHoSo"],
                        "PlateletsTruoc":ban_dau["platelets"]})
nhat_ky["Nhom"] = nhat_ky["PlateletsTruoc"].map(xac_dinh_nhom)
dong_nhom = []
for nhom in range(1, so_nhom + 1):
    gia_tri = nhat_ky.loc[nhat_ky["Nhom"].eq(nhom), "PlateletsTruoc"]
    mean_nhom = None
    gia_tri_sau = None
    if len(gia_tri) > 0:
        with localcontext() as ctx:
            ctx.prec = 38
            mean_nhom = (sum(gia_tri, Decimal(0)) / len(gia_tri)).quantize(
                Decimal("0.0000000001"), rounding=ROUND_HALF_UP)
            gia_tri_sau = mean_nhom.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
    dong_nhom.append({"Nhom":nhom, "CanDuoi":min_p+(nhom-1)*do_rong,
        "CanTren":min_p+nhom*do_rong, "SoHoSo":len(gia_tri),
        "MeanNhom":mean_nhom, "PlateletsSau":gia_tri_sau})
thong_ke_nhom = pd.DataFrame(dong_nhom)
tra_nhom = thong_ke_nhom.set_index("Nhom")
nhat_ky["MeanNhom"] = nhat_ky["Nhom"].map(tra_nhom["MeanNhom"])
nhat_ky["PlateletsSau"] = nhat_ky["Nhom"].map(tra_nhom["PlateletsSau"])
nhat_ky = nhat_ky[["MaHoSo", "Nhom", "PlateletsTruoc", "MeanNhom", "PlateletsSau"]]
ban_lam_min = ban_dau.copy(deep=True)
ban_lam_min["platelets"] = nhat_ky["PlateletsSau"]
tong_ket = pd.DataFrame({"SoHoSo":[len(ban_lam_min)], "SoNhom":[so_nhom],
    "DoRongNhom":[do_rong], "SoHoSoDaLamMin":[nhat_ky["PlateletsSau"].notna().sum()],
    "SoGiaTriThayDoi":[nhat_ky["PlateletsTruoc"].ne(nhat_ky["PlateletsSau"]).sum()]})
tong_ket

,SoHoSo,SoNhom,DoRongNhom,SoHoSoDaLamMin,SoGiaTriThayDoi
0,299,5,164980.000000,299,299


In [3]:
thong_ke_nhom

,Nhom,CanDuoi,CanTren,SoHoSo,MeanNhom,PlateletsSau
0,1,25100.000000,190080.000000,51,142198.0392156863,142198.04
1,2,190080.000000,355060.000000,210,261884.0380000000,261884.04
2,3,355060.000000,520040.000000,34,408617.6470588235,408617.65
3,4,520040.000000,685020.000000,2,582000.0000000000,582000.00
4,5,685020.000000,850000.000000,2,796000.0000000000,796000.00


## 3. Mười ví dụ trước sau
Chọn hai hồ sơ mỗi nhóm. MaHoSo là số thứ tự nội bộ; SQL và Python có thể đánh số khác nhau. Đối chiếu bằng nhóm, platelets trước, mean và giá trị sau.

In [4]:
vi_du = (nhat_ky.sort_values(["Nhom", "PlateletsTruoc", "MaHoSo"])
    .groupby("Nhom", sort=False).head(2).reset_index(drop=True))
vi_du

,MaHoSo,Nhom,PlateletsTruoc,MeanNhom,PlateletsSau
0,278,1,25100.000000,142198.0392156863,142198.04
1,16,1,47000.000000,142198.0392156863,142198.04
2,166,2,192000.000000,261884.0380000000,261884.04
3,45,2,194000.000000,261884.0380000000,261884.04
4,217,3,358000.000000,408617.6470588235,408617.65
5,32,3,360000.000000,408617.6470588235,408617.65
6,288,4,543000.000000,582000.0000000000,582000.00
7,106,4,621000.000000,582000.0000000000,582000.00
8,297,5,742000.000000,796000.0000000000,796000.00
9,110,5,850000.000000,796000.0000000000,796000.00


**Nhận xét:** 5 nhóm có 51, 210, 34, 2, 2 hồ sơ, tổng 299. Giá trị thay thế lần lượt là 142.198,04; 261.884,04; 408.617,65; 582.000,00; 796.000,00. Các cột khác giữ nguyên. Chưa thực hiện phần tăng/giảm 10%.

## 4. Lưu kết quả làm mịn
Lưu nhật ký đủ 299 hồ sơ, thống kê nhóm, ví dụ và bản làm việc trung gian; không ghi đè dữ liệu đầu vào hoặc tạo after cuối.

In [5]:
thu_muc_kq = thu_muc / "ket_qua" / "13_lam_min" / "02_mean_5_nhom"
thu_muc_kq.mkdir(parents=True, exist_ok=True)
tong_ket.to_csv(thu_muc_kq / "tong_ket_python.csv", index=False, encoding="utf-8-sig")
thong_ke_nhom.to_csv(thu_muc_kq / "nhom_python.csv", index=False, encoding="utf-8-sig")
nhat_ky.to_csv(thu_muc_kq / "nhat_ky_python.csv", index=False, encoding="utf-8-sig")
vi_du.to_csv(thu_muc_kq / "vi_du_python.csv", index=False, encoding="utf-8-sig")
ban_lam_min.to_csv(thu_muc_kq / "ban_lam_viec_sau_mean_python.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"TrangThai":["Đã lưu kết quả mean 5 nhóm; giữ nguyên dữ liệu đầu vào."]})

,TrangThai
0,Đã lưu kết quả mean 5 nhóm; giữ nguyên dữ liệu...
